## Evaluate Generative Retrieval Model (T5)

Evaluates the trained T5 seq2seq model with beam search retrieval.

**What this notebook does:**
1. Load trained model checkpoint
2. Beam-search decode top-K Semantic IDs per user
3. Map Semantic IDs to items, filter invalid IDs
4. Compute Recall@K, NDCG@K on val + test splits
5. Compare against paper Table 1 (Toys & Games)

In [24]:
import sys

if "../" not in sys.path:
    sys.path.insert(0, "../")

from pathlib import Path

import torch
from transformers import T5ForConditionalGeneration

from tiger.dataset import TigerDataset
from tiger.evaluation import evaluate
from tiger.utils import get_device, set_seed

In [27]:
# Paths
DATA_DIR = Path("../data/2014/processed")
SPLITS_PATH = DATA_DIR / "splits.parquet"
SEMANTIC_IDS_PATH = Path("../checkpoints/rqvae/semantic_ids.pt")
CHECKPOINT_PATH = Path("../checkpoints/model/checkpoint-100000")

# Evaluation config
BATCH_SIZE = 64  # beam search is memory-heavy; keep modest
BEAM_SIZE = 20  # matches paper's Fig. 6 (top-20 retrieval)
AT_K = [5, 10]  # paper evaluates K = 5, 10

SEED = 42

In [28]:
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

2026-09-13 20:00:48.460 | INFO     | tiger.utils:set_seed:27 - Random seed set to 42
2026-09-13 20:00:48.464 | INFO     | tiger.utils:get_device:16 - Using device: mps


Device: mps


Load sid_to_asin mapping

In [ ]:
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

print(f"Items with Semantic IDs: {len(sid_to_asin):,}")

Items with Semantic IDs: 11,924


Load trained T5 model from checkpoint

In [31]:
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_PATH).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

Total parameters: 14,540,160


### Evaluate Validation Split

Val users have their full training history as input and the held-out
second-to-last item as target.

In [33]:
val_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="val",
    max_seq_len=20,
)

print(f"Val users: {len(val_dataset):,}")

val_results = evaluate(
    model=model,
    dataset=val_dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=BATCH_SIZE,
    beam_size=BEAM_SIZE,
    at_k=AT_K,
)

print("\nVal results:")
for k, v in val_results.items():
    print(f"  {k}: {v:.4f}")

Val users: 19,412


Evaluating: 100%|██████████| 304/304 [02:30<00:00,  2.02it/s]


Val results:
  recall@5: 0.0087
  ndcg@5: 0.0047
  recall@10: 0.0164
  ndcg@10: 0.0072


### Evaluate Test Split

Test users have training history + val item as input and the held-out
last item as target. This matches the paper's final reported numbers.

In [34]:
test_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="test",
    max_seq_len=20,
)

print(f"Test users: {len(test_dataset):,}")

test_results = evaluate(
    model=model,
    dataset=test_dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=BATCH_SIZE,
    beam_size=BEAM_SIZE,
    at_k=AT_K,
)

print("\nTest results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

Test users: 19,412


Evaluating: 100%|██████████| 304/304 [02:25<00:00,  2.09it/s]


Test results:
  recall@5: 0.0057
  ndcg@5: 0.0031
  recall@10: 0.0104
  ndcg@10: 0.0047


### Compare with Paper (Toys & Games, Table 1)

In [35]:
import pandas as pd

paper = {
    "recall@5": 0.0521,
    "ndcg@5": 0.0371,
    "recall@10": 0.0712,
    "ndcg@10": 0.0432,
}

paper_seeds = {
    "recall@5": (0.0518, 0.00064),
    "ndcg@5": (0.0375, 0.00039),
    "recall@10": (0.0698, 0.0013),
    "ndcg@10": (0.0433, 0.00047),
}

sasrec_best_baseline = {
    "recall@5": 0.0463,
    "ndcg@5": 0.0306,
    "recall@10": 0.0675,
    "ndcg@10": 0.0374,
}

df = pd.DataFrame(
    {
        "Ours (val)": val_results,
        "Ours (test)": test_results,
        "Paper TIGER": paper,
        "Paper (3 seeds)": {
            k: f"{m:.4f} ± {s:.4f}" for k, (m, s) in paper_seeds.items()
        },
        "SASRec baseline": sasrec_best_baseline,
    }
)
df.index.name = "Metric"
df.round(4)

,Ours (val),Ours (test),Paper TIGER,Paper (3 seeds),SASRec baseline
Metric,,,,,
recall@5,0.0087,0.0057,0.0521,0.0518 ± 0.0006,0.0463
ndcg@5,0.0047,0.0031,0.0371,0.0375 ± 0.0004,0.0306
recall@10,0.0164,0.0104,0.0712,0.0698 ± 0.0013,0.0675
ndcg@10,0.0072,0.0047,0.0432,0.0433 ± 0.0005,0.0374


In [36]:
import json, torch
from pathlib import Path
from transformers import T5ForConditionalGeneration
from tiger.dataset import TigerDataset
from tiger.evaluation import beam_search_retrieval, decode_semantic_ids
from tiger.utils import get_device

# 1. Training convergence
state = json.load(open(CHECKPOINT_PATH / "trainer_state.json"))
print("=== TRAINING STATE ===")
print(f'Final step: {state["global_step"]}')
print(f'Last eval loss: {state["log_history"][-1]}')

# 2. Generation config sanity
gen = json.load(open(CHECKPOINT_PATH / "generation_config.json"))
print()
print("=== GENERATION CONFIG ===")
for k, v in gen.items():
    print(f"  {k}: {v}")

# 3. Beam diversity check on 20 users
model.eval()
device = get_device()
model.to(device)

dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="test",
)
dataset.samples = dataset.samples[:20]

from torch.utils.data import DataLoader

batch = next(iter(DataLoader(dataset, batch_size=20, shuffle=False)))

with torch.no_grad():
    generated = beam_search_retrieval(
        model,
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        beam_size=20,
    )

preds = decode_semantic_ids(generated.cpu(), sid_to_asin)
distinct = [len(p) for p in preds]
print()
print("=== BEAM DIVERSITY (20 users, beam=20) ===")
print(f"Avg distinct valid items per user: {sum(distinct)/len(distinct):.1f} / 20")
print(f"Per-user distinct counts: {distinct}")

2026-09-13 20:20:54.312 | INFO     | tiger.utils:get_device:16 - Using device: mps


=== TRAINING STATE ===
Final step: 100000
Last eval loss: {'epoch': 233.6448598130841, 'eval_loss': 2.384739398956299, 'eval_runtime': 2.2256, 'eval_samples_per_second': 8721.993, 'eval_sequence_accuracy': 0.0019575520296723674, 'eval_steps_per_second': 34.148, 'eval_token_accuracy': 0.4885508963527715, 'step': 100000}

=== GENERATION CONFIG ===
  _from_model_config: True
  decoder_start_token_id: 3024
  eos_token_id: 3025
  output_attentions: False
  output_hidden_states: False
  pad_token_id: 3024
  transformers_version: 5.8.1
  use_cache: True

=== BEAM DIVERSITY (20 users, beam=20) ===
Avg distinct valid items per user: 20.0 / 20
Per-user distinct counts: [20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20]


In [ ]:
state = json.load(open(CHECKPOINT_PATH / "trainer_state.json"))
history = state["log_history"]

# Print eval loss at each eval step
evals = [h for h in history if "eval_loss" in h]
print("step    eval_loss   token_acc   seq_acc")
for h in evals:
    print(
        f"{h['step']:>6}  {h['eval_loss']:.4f}    {h.get('eval_token_accuracy', 0):.4f}    {h.get('eval_sequence_accuracy', 0):.4f}"
    )

# Also last few training losses
trains = [h for h in history if "loss" in h and "eval_loss" not in h]
print()
print("last 5 train losses:")
for h in trains[-5:]:
    print(f"  step {h['step']}: {h['loss']:.4f}")

step    eval_loss   token_acc   seq_acc
  5000  4.0414    0.2640    0.0001
 10000  3.6836    0.3067    0.0009
 15000  3.3185    0.3571    0.0006
 20000  3.0659    0.3909    0.0009
 25000  2.8661    0.4174    0.0003
 30000  2.7811    0.4295    0.0011
 35000  2.6609    0.4488    0.0005
 40000  2.6262    0.4540    0.0002
 45000  2.5753    0.4628    0.0006
 50000  2.5095    0.4725    0.0013
 55000  2.4875    0.4777    0.0016
 60000  2.4597    0.4790    0.0018
 65000  2.4446    0.4820    0.0017
 70000  2.4416    0.4818    0.0015
 75000  2.4217    0.4852    0.0011
 80000  2.4114    0.4859    0.0012
 85000  2.4105    0.4843    0.0012
 90000  2.4016    0.4859    0.0022
 95000  2.4007    0.4880    0.0021
100000  2.3847    0.4886    0.0020

last 5 train losses:
  step 99600: 2.3739
  step 99700: 2.3704
  step 99800: 2.3679
  step 99900: 2.3679
  step 100000: 2.3632


In [38]:
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_PATH)
model.eval()

dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="test",
)
dataset.samples = dataset.samples[:500]

device = get_device()

for beam in [20, 50, 100]:
    results = evaluate(
        model=model,
        dataset=dataset,
        sid_to_asin=sid_to_asin,
        device=device,
        batch_size=16,
        beam_size=beam,
        at_k=[5, 10],
    )
    print(f"beam={beam:>3}: ", {k: round(v, 4) for k, v in results.items()})

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

2026-09-13 20:25:24.265 | INFO     | tiger.utils:get_device:16 - Using device: mps
Evaluating: 100%|██████████| 32/32 [00:05<00:00,  5.66it/s]


beam= 20:  {'recall@5': 0.004, 'ndcg@5': 0.0019, 'recall@10': 0.012, 'ndcg@10': 0.0043}


Evaluating: 100%|██████████| 32/32 [00:09<00:00,  3.20it/s]


beam= 50:  {'recall@5': 0.004, 'ndcg@5': 0.0019, 'recall@10': 0.016, 'ndcg@10': 0.0055}


Evaluating: 100%|██████████| 32/32 [00:18<00:00,  1.78it/s]

beam=100:  {'recall@5': 0.002, 'ndcg@5': 0.0008, 'recall@10': 0.016, 'ndcg@10': 0.0052}


In [39]:
model.eval()

# Train split — the model was directly trained on these exact (history, target) pairs
dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="train",
    sliding_window=True,
)
dataset.samples = dataset.samples[:500]

device = get_device()

results = evaluate(
    model=model,
    dataset=dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=16,
    beam_size=50,
    at_k=[5, 10],
)
print("TRAIN-SPLIT RESULTS (500 samples the model was trained on):")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

2026-09-13 20:40:29.699 | INFO     | tiger.utils:get_device:16 - Using device: mps
Evaluating: 100%|██████████| 32/32 [00:15<00:00,  2.00it/s]

TRAIN-SPLIT RESULTS (500 samples the model was trained on):
  recall@5: 0.0120
  ndcg@5: 0.0074
  recall@10: 0.0200
  ndcg@10: 0.0100


In [40]:
"""Overfit test: can a fresh T5 memorize 300 users' training examples?"""

from torch.utils.data import DataLoader
from tqdm import tqdm

from tiger.model import create_model

set_seed(42)
device = get_device()

# --- 1. Tiny training set: 300 users' sliding-window examples ---
dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="train",
    sliding_window=True,
)

# Keep only examples from the first 300 users' worth of samples (~1700 examples)
n_samples = 1700
dataset.samples = dataset.samples[:n_samples]
print(f"Training examples: {len(dataset)}")

# --- 2. Fresh model, same architecture as paper ---
model = create_model(vocab_size=dataset.vocab_size, pad_token_id=dataset.pad_token).to(
    device
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

# --- 3. Train for 3000 steps (≈ 226 epochs over 1700 examples) ---
loader = DataLoader(dataset, batch_size=128, shuffle=True)

model.train()
steps = 0
iterator = iter(loader)
pbar = tqdm(total=3000, desc="Overfitting")
while steps < 3000:
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)

    batch = {k: v.to(device) for k, v in batch.items()}
    loss = model(**batch).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    steps += 1
    if steps % 300 == 0:
        pbar.set_postfix(loss=f"{loss.item():.3f}")
    pbar.update(1)

print(f"\nFinal train loss: {loss.item():.4f}")

# --- 4. Evaluate on the SAME examples ---
model.eval()
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

results = evaluate(
    model=model,
    dataset=dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=128,
    beam_size=50,
    at_k=[5, 10],
)
print("\nOVERFIT RESULTS (same 1700 examples the model was trained on):")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

2026-09-13 20:46:49.292 | INFO     | tiger.utils:set_seed:27 - Random seed set to 42
2026-09-13 20:46:49.299 | INFO     | tiger.utils:get_device:16 - Using device: mps


Training examples: 1700


Overfitting: 100%|██████████| 3000/3000 [13:33<00:00,  3.12it/s, loss=3.987]


Final train loss: 3.9875


Evaluating: 100%|██████████| 14/14 [01:31<00:00,  6.56s/it]


OVERFIT RESULTS (same 1700 examples the model was trained on):
  recall@5: 0.0047
  ndcg@5: 0.0038
  recall@10: 0.0047
  ndcg@10: 0.0038
